In [15]:
import pandas as pd
import numpy as np 
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import MNIST
import matplotlib.pyplot as plt
import torch.optim as optim
import seaborn as sb
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix 
from sklearn.model_selection import train_test_split
import torchvision.transforms as transforms

In [11]:
# Transform the data
transform = transforms.Compose([
    transforms.ToTensor(), # this will not only transform the image to the tensor but also rescale it (0 to 1)
    transforms.Normalize((0.1307,), (0.3081,))
])

train_set = MNIST(root = "./Assig_data", train = True, download=True, transform=transform)
test_set = MNIST(root = "./Assig_data", train = False, download = True, transform = transform)

In [12]:
# define the dataloader
training_Loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_Loader = DataLoader(test_set, batch_size=64)


# Build the CNN model


In [18]:
class CNN(nn.Module):
    def __init__(self):
        super (CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            # first layer 
            nn.Conv2d(1,28, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel and stride values are 2 and 2 

            # second layer
                
             nn.Conv2d(28,56, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
                
            # third layer
                
             nn.Conv2d(56,84, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fcc_layer = nn.Sequential(
            nn.Linear(3*3*84, 256), # 256 is random value for neuron
            nn.ReLU(),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        x= self.conv_layers(x)
        x= x.view(x.size(0), -1) 
        # latening # .view() reshape tensor into shape (a, b), # x.size(0) So you keep batch size unchanged. # -1 figure out this dimension automatically
        x= self.fcc_layer(x)
        return x 

In [19]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Train the model

In [20]:
epocs = 10

for epoc in range(epocs):
    model.train()
    training_loss = 0.0
    for image, label in training_Loader:
        optimizer.zero_grad()
        outputs = model.forward(image)
        loss = criterion(outputs, label)
        loss.backward()
        optimizer.step() # update parameter
        training_loss += loss.item()
    print(f" {epoc}/{epocs} the training loss >>>> {training_loss}")
    

 0/10 the training loss >>>> 141.40651523612905
 1/10 the training loss >>>> 39.679511054535396
 2/10 the training loss >>>> 29.19692084629787
 3/10 the training loss >>>> 22.976020195230376
 4/10 the training loss >>>> 18.623828390511335
 5/10 the training loss >>>> 15.81930569873657
 6/10 the training loss >>>> 13.13166066051599
 7/10 the training loss >>>> 11.201539593815141
 8/10 the training loss >>>> 9.482039355384586
 9/10 the training loss >>>> 9.056661568825916


# Evaluate the Model 

In [23]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for image, label in test_Loader:
        outputs = model(image)
        _, predicted = torch.max(outputs, 1)

        correct += (predicted== label).sum().item()
        total += label.size(0)
    print(f" The accuracy of the model is>> {(correct/total) *100} %")

 The accuracy of the model is>> 99.24 %
